# Noble AI · GPU Worker en Google Colab

Este notebook configura **llama.cpp (CUDA)**, un GGUF ligero, el proxy protegido de Noble AI y un túnel Cloudflare para conectar tu aplicación de escritorio con la GPU de Colab.

**Antes:** abre *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*. Ejecuta las celdas en orden. Ningún modelo se descarga hasta que ejecutes estas celdas. La interfaz de Noble AI sigue siendo la aplicación web local de tu PC.

**Importante:** el token generado es secreto; no publiques este notebook con las salidas visibles. La sesión de Colab puede finalizar, el túnel es temporal y las mediciones solo son válidas después de ejecutar realmente este entorno.


In [ ]:
import os, sys, subprocess, shutil, time, json, secrets, re
from pathlib import Path
from urllib.request import urlopen, Request, urlretrieve

BASE = Path("/content/noble_worker")
BASE.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Este notebook necesita un entorno Colab con GPU NVIDIA activada.")
subprocess.run(["nvidia-smi"], check=True)
print("NVCC disponible:", bool(shutil.which("nvcc")))


## 1. Descargar Noble AI e instalar las dependencias del proxy

Solo se instala el backend del worker; no necesitas construir React en Colab.


In [ ]:
NOBLE_REPO = BASE / "novel_ai"
if not (NOBLE_REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Mairon31/novel_ai.git", str(NOBLE_REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(NOBLE_REPO), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", str(NOBLE_REPO / "apps/api/requirements.txt"),
                "huggingface_hub>=0.27,<2"], check=True)
commit = subprocess.check_output(["git", "-C", str(NOBLE_REPO), "rev-parse", "--short=12", "HEAD"], text=True).strip()
print("Noble AI commit:", commit)


## 2. Compilar llama.cpp con CUDA

Se utiliza el servidor OpenAI-compatible oficial de llama.cpp. Compilar desde fuente permite aprovechar distintas GPUs que Colab asigne (T4, L4 u otras).


In [ ]:
if not shutil.which("cmake") or not shutil.which("nvcc"):
    raise RuntimeError("Falta cmake o nvcc en este runtime. Revisa que Colab tenga CUDA activo.")
LLAMA_REPO = BASE / "llama.cpp"
if not (LLAMA_REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(LLAMA_REPO), "pull", "--ff-only"], check=True)
subprocess.run(["cmake", "-S", str(LLAMA_REPO), "-B", str(LLAMA_REPO / "build"),
                "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"], check=True)
subprocess.run(["cmake", "--build", str(LLAMA_REPO / "build"), "-j", "2", "--target", "llama-server"], check=True)
LLAMA_SERVER = LLAMA_REPO / "build/bin/llama-server"
if not LLAMA_SERVER.is_file():
    raise RuntimeError("No se encontró llama-server después de compilar.")
print("Binario listo:", LLAMA_SERVER)


## 3. Seleccionar un GGUF para el primer ensayo

Por defecto se utiliza **Qwen3-4B-Instruct-2507 Q4_K_M** (aprox. 2,5 GB de pesos). Es un modelo pequeño para verificar el transporte; no pretende representar la calidad final de Noble AI. Puedes modificar repositorio y archivo antes de ejecutar.


In [ ]:
from huggingface_hub import hf_hub_download, HfApi

HF_REPO = "bartowski/Qwen_Qwen3-4B-Instruct-2507-GGUF"
HF_FILE = "Qwen_Qwen3-4B-Instruct-2507-Q4_K_M.gguf"
MODEL_DIR = BASE / "models"
MODEL_DIR.mkdir(exist_ok=True)
# Check model files from the live HF repo before downloading.
repo_info = HfApi().model_info(HF_REPO)
model_rev = repo_info.sha
available = [item.rfilename for item in repo_info.siblings if item.rfilename.endswith(".gguf")]
if HF_FILE not in available:
    candidates = [item for item in available if item.endswith("-Q4_K_M.gguf")]
    if len(candidates) != 1:
        raise FileNotFoundError("GGUF Q4_K_M no disponible; catálogo: " + str(available[:25]))
    HF_FILE = candidates[0]
    print("Nombre real del GGUF:", HF_FILE)
MODEL_PATH = Path(hf_hub_download(repo_id=HF_REPO, filename=HF_FILE,
                                  revision=model_rev, local_dir=str(MODEL_DIR)))
print("Modelo:", HF_REPO, "archivo:", MODEL_PATH.name)
print("Revisión HF:", model_rev)
print("Tamaño (MB):", round(MODEL_PATH.stat().st_size / (1024**2), 1))


## 4. Iniciar el motor LLM

La API de llama.cpp escucha únicamente en loopback, puerto 5001. No la publiques directamente.


In [ ]:
ENGINE_LOG = (BASE / "llama-server.log").open("ab", buffering=0)
ENGINE_CMD = [str(LLAMA_SERVER), "--model", str(MODEL_PATH),
              "--host", "127.0.0.1", "--port", "5001",
              "--ctx-size", "4096", "--n-gpu-layers", "99", "--parallel", "1"]
if "engine_process" not in globals() or engine_process.poll() is not None:
    engine_process = subprocess.Popen(ENGINE_CMD, stdout=ENGINE_LOG,
                                      stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)
for _ in range(90):
    if engine_process.poll() is not None:
        raise RuntimeError("llama-server terminó; revisa /content/noble_worker/llama-server.log")
    try:
        with urlopen("http://127.0.0.1:5001/v1/models", timeout=2) as response:
            models = json.load(response)
        if models.get("data"):
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("El servidor no respondió a /v1/models; revisa los logs.")
print("Motor listo. Modelos:", [m.get("id") for m in models.get("data", [])])


## 5. Iniciar el proxy autenticado

Se crea un token aleatorio por sesión. **Guárdalo para introducirlo en Configuración → Red y workers de Noble AI.** No se escribe en un archivo de configuración ni en el repositorio.


In [ ]:
WORKER_TOKEN = secrets.token_urlsafe(36)
worker_env = dict(os.environ)
worker_env["NOBLE_WORKER_TOKEN"] = WORKER_TOKEN
worker_env["NOBLE_WORKER_UPSTREAM"] = "http://127.0.0.1:5001"
WORKER_LOG = (BASE / "noble-worker.log").open("ab", buffering=0)
if "proxy_process" not in globals() or proxy_process.poll() is not None:
    proxy_process = subprocess.Popen([
        sys.executable, "-m", "uvicorn", "noble_api.worker_server:app",
        "--app-dir", str(NOBLE_REPO / "apps/api"), "--host", "127.0.0.1",
        "--port", "8765",
    ], env=worker_env, stdout=WORKER_LOG, stderr=subprocess.STDOUT,
       stdin=subprocess.DEVNULL)
for _ in range(30):
    try:
        req = Request("http://127.0.0.1:8765/v1/health",
                      headers={"Authorization": "Bearer " + WORKER_TOKEN})
        with urlopen(req, timeout=3) as res:
            health = json.load(res)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Proxy sin respuesta; revisa /content/noble_worker/noble-worker.log")
print("Proxy autorizado:", health["protocol"], "| motor:", health["engine_status"])
print("TOKEN PARA NOBLE AI (no publicar):", WORKER_TOKEN)


## 6. Crear el túnel Cloudflare hacia el proxy (no hacia llama.cpp)

El túnel HTTPS apunta a `127.0.0.1:8765`, que **exige token**. Nunca expongas el puerto 5001 sin autenticación.


In [ ]:
CLOUDFLARED_DEB = BASE / "cloudflared.deb"
if not shutil.which("cloudflared"):
    urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
                str(CLOUDFLARED_DEB))
    subprocess.run(["dpkg", "-i", str(CLOUDFLARED_DEB)], check=True)
TUNNEL_LOG_PATH = BASE / "cloudflared.log"
TUNNEL_LOG = TUNNEL_LOG_PATH.open("ab", buffering=0)
if "tunnel_process" not in globals() or tunnel_process.poll() is not None:
    tunnel_process = subprocess.Popen([
        "cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8765"
    ], stdout=TUNNEL_LOG, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)
TUNNEL_URL = None
for _ in range(45):
    time.sleep(1)
    text = TUNNEL_LOG_PATH.read_text(errors="replace")
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", text, flags=re.I)
    if found:
        TUNNEL_URL = found[-1]
        break
    if tunnel_process.poll() is not None:
        raise RuntimeError("cloudflared terminó. Revisa cloudflared.log")
if not TUNNEL_URL:
    raise RuntimeError("No se detectó una URL trycloudflare.com; revisa cloudflared.log")
print("URL para Noble AI:", TUNNEL_URL)
print("Token para Noble AI (no publicar):", WORKER_TOKEN)
print("Nombre sugerido: GPU Colab")


## 7. Verificar la conexión y medir rendimiento

Las comprobaciones se realizan contra la URL HTTPS real del túnel, no contra un servidor simulado. El JSON final guarda latencias y VRAM observada, sin registrar texto narrativo ni la clave.


In [ ]:
BENCHMARK_REPORT = BASE / "noble-worker-benchmark.json"
subprocess.run([
    sys.executable, str(NOBLE_REPO / "tools/worker_benchmark.py"),
    "--url", TUNNEL_URL, "--runs", "3", "--timeout", "180",
    "--output", str(BENCHMARK_REPORT),
], env=worker_env, check=True)
report = json.loads(BENCHMARK_REPORT.read_text())
print(json.dumps({k: report[k] for k in [
    "model", "runs_succeeded", "runs_failed", "ttft_p50_ms",
    "ttft_p95_ms", "gpu_peak_used_mb", "gpu_peak_utilization_percent",
]}, indent=2))
print("Ruta de evidencia:", BENCHMARK_REPORT)


## 8. Detener el ensayo

Cuando termines de utilizar Colab, esta celda detiene **únicamente** los procesos iniciados por este notebook. El equipo principal conservará las historias y tendrás que volver a registrar o actualizar la URL si Colab inicia un túnel nuevo.


In [ ]:
for process_name in ("tunnel_process", "proxy_process", "engine_process"):
    proc = globals().get(process_name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
        print("Detenido:", process_name)
